[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/time-travel.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239536-lesson-5-time-travel)

# 时间旅行

## 回顾

我们讨论了人工参与循环的动机：

(1) `批准` - 我们可以中断我们的agent，向用户显示状态，并允许用户接受某个行动

(2) `调试` - 我们可以回退图以重现或避免问题

(3) `编辑` - 你可以修改状态

我们展示了断点如何在特定节点停止图或允许图动态中断自己。

然后我们展示了如何通过人工批准继续或直接使用人工反馈编辑图状态。

## 目标

现在，让我们展示LangGraph如何通过查看、重放甚至从过去状态分叉来[支持调试](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/time-travel/)。

我们称之为`时间旅行`。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_openai langgraph_sdk langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

让我们构建我们的agent。

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """将a和b相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """将a和b相加。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """将a除以b。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="您是一个有用的助手，负责对一组输入执行算术运算。")

# 节点
def assistant(state: MessagesState):
   """助手节点，处理消息并调用LLM"""
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些做实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些确定控制流
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自assistant的最新消息（结果）是工具调用 -> tools_condition路由到tools
    # 如果来自assistant的最新消息（结果）不是工具调用 -> tools_condition路由到END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(checkpointer=MemorySaver())

# 显示
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

让我们像之前一样运行它。

In [ ]:
# 输入
initial_input = {"messages": HumanMessage(content="计算2乘以3")}

# 线程
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

## 浏览历史

我们可以使用`get_state`来查看我们图的**当前**状态，给定`thread_id`！

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

我们也可以浏览我们agent的状态历史。

`get_state_history`让我们获取所有先前步骤的状态。

In [ ]:
all_states = [s for s in graph.get_state_history(thread)]

In [ ]:
len(all_states)

第一个元素是当前状态，就像我们从`get_state`得到的一样。

In [ ]:
all_states[-2]

上面的所有内容我们都可以在这里可视化：

![fig1.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb038211b544898570be3_time-travel1.png)

## 重放

我们可以从任何先前的步骤重新运行我们的agent。

![fig2.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb038a0bd34b541c78fb8_time-travel2.png)

让我们回到接收人工输入的步骤！

In [ ]:
to_replay = all_states[-2]

In [ ]:
to_replay

查看状态。

In [ ]:
to_replay.values

我们可以看到要调用的下一个节点。

In [ ]:
to_replay.next

我们还获得了配置，它告诉我们`checkpoint_id`以及`thread_id`。

In [ ]:
to_replay.config

要从这里重放，我们只需将配置传递回agent！

图知道这个检查点已经被执行过。

它只是从这个检查点重放！

In [ ]:
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，我们可以看到agent重新运行后的当前状态。

## 分叉

如果我们想要从同一个步骤运行，但使用不同的输入怎么办。

这就是分叉。

![fig3.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb038f89f2d847ee5c336_time-travel3.png)

In [ ]:
to_fork = all_states[-2]
to_fork.values["messages"]

同样，我们有配置。

In [ ]:
to_fork.config

让我们在这个检查点修改状态。

我们可以只运行`update_state`并提供`checkpoint_id`。

记住我们的`messages`减速器是如何工作的：

* 它会附加，除非我们提供消息ID。
* 我们提供消息ID来覆盖消息，而不是附加到状态！

所以，要覆盖消息，我们只需提供消息ID，我们有`to_fork.values["messages"].id`。

In [ ]:
fork_config = graph.update_state(
    to_fork.config,
    {"messages": [HumanMessage(content='计算5乘以3', 
                               id=to_fork.values["messages"][0].id)]},
)

In [ ]:
fork_config

这创建了一个新的、分叉的检查点。

但是，元数据 - 例如，下一步去哪里 - 被保留了！

我们可以看到我们agent的当前状态已经用我们的分叉更新了。

In [ ]:
all_states = [state for state in graph.get_state_history(thread) ]
all_states[0].values["messages"]

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

现在，当我们流式传输时，图知道这个检查点从未被执行过。

所以，图运行，而不是简单地重放。

In [ ]:
for event in graph.stream(None, fork_config, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，我们可以看到当前状态是我们agent运行的结束。

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

### 与LangGraph API的时间旅行

**⚠️ 免责声明**

自从拍摄这些视频以来，我们更新了Studio，使其可以在本地运行并在浏览器中打开。这现在是运行Studio的首选方式（而不是像视频中显示的那样使用桌面应用程序）。请参阅[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)的本地开发服务器文档和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)的相关说明。要启动本地开发服务器，请在此模块的`/studio`目录中的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

我们通过SDK连接到它并展示LangGraph API如何[支持时间旅行](https://langchain-ai.github.io/langgraph/cloud/how-tos/human_in_the_loop_time_travel/#initial-invocation)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("很抱歉，Google Colab目前不支持LangGraph Studio")

In [ ]:
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

#### 重放

让我们运行我们的agent，流式传输每个节点调用后图状态的`updates`。

In [ ]:
initial_input = {"messages": HumanMessage(content="计算2乘以3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id = "agent",
    input=initial_input,
    stream_mode="updates",
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"助手节点"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"工具节点"+"-" * 20)
            print(tool_node[-1])

现在，让我们看看从指定检查点**重放**。

我们只需要传递`checkpoint_id`。

In [ ]:
states = await client.threads.get_history(thread['thread_id'])
to_replay = states[-2]
to_replay

让我们用`stream_mode="values"`流式传输，以便在重放时查看每个节点的完整状态。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="values",
    checkpoint_id=to_replay['checkpoint_id']
):      
    print(f"接收到类型为: {chunk.event}的新事件...")
    print(chunk.data)
    print("\n\n")

我们可以将这些全部视为仅流式传输我们重放的节点对状态所做的`updates`。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="updates",
    checkpoint_id=to_replay['checkpoint_id']
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"助手节点"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"工具节点"+"-" * 20)
            print(tool_node[-1])

#### 分叉

现在，让我们看看分叉。

让我们获取与上面相同的步骤，即人工输入。

让我们用我们的agent创建一个新线程。

In [ ]:
initial_input = {"messages": HumanMessage(content="计算2乘以3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="updates",
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"助手节点"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"工具节点"+"-" * 20)
            print(tool_node[-1])

In [ ]:
states = await client.threads.get_history(thread['thread_id'])
to_fork = states[-2]
to_fork['values']

In [ ]:
to_fork['values']['messages'][0]['id']

In [ ]:
to_fork['next']

In [ ]:
to_fork['checkpoint_id']

让我们编辑状态。

记住我们的`messages`减速器是如何工作的：

* 它会附加，除非我们提供消息ID。
* 我们提供消息ID来覆盖消息，而不是附加到状态！

In [ ]:
forked_input = {"messages": HumanMessage(content="计算3乘以3",
                                         id=to_fork['values']['messages'][0]['id'])}

forked_config = await client.threads.update_state(
    thread["thread_id"],
    forked_input,
    checkpoint_id=to_fork['checkpoint_id']
)

In [ ]:
forked_config

In [ ]:
states = await client.threads.get_history(thread['thread_id'])
states[0]

要重新运行，我们传入`checkpoint_id`。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="updates",
    checkpoint_id=forked_config['checkpoint_id']
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"助手节点"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"工具节点"+"-" * 20)
            print(tool_node[-1])

### LangGraph Studio

让我们在Studio UI中查看我们的`agent`的分叉，它使用`module-1/studio/langgraph.json`中设置的`module-1/studio/agent.py`。